# 09 - Ensemble

Combines the predictions of notebooks 05-08 with a weighted average of their `log1p(price)` predictions.

Two questions:
1. Does a blend beat the best single model?
2. Does the text (TF-IDF in 06, BamiBERT in 08) add information the structured columns do not have?

The weights are learned on the validation set and the blends are scored on the test set.
Run notebooks 05-08 first: they save their predictions to `reports/predictions/`.

In [1]:
import os

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             r2_score, root_mean_squared_error)

## Data

One column per model; the rows (index) are the same validation and test listings in every file.

In [2]:
files = ["05_tree_ensembles", "06_text_tfidf", "07_deep_tabular", "08_bamibert"]
pred = pd.concat([pd.read_parquet(f"../reports/predictions/{name}.parquet").drop(columns="split")
                  for name in files], axis=1)

data = pd.read_parquet("../data/processed/model_input.parquet", columns=["price", "split"])
pred = pred.join(data)

val = pred[pred["split"] == "val"]
test = pred[pred["split"] == "test"]

pred.head()

,LightGBM,CatBoost,XGBoost,Ridge text + structured,LightGBM + text,MLP,BamiBERT,price,split
1677900,22.836163,21.655833,22.528944,22.887657,22.326154,22.841642,22.069439,5.000000e+09,val
1677901,22.274835,22.251559,22.247770,22.249714,22.238223,22.278290,22.354562,4.300000e+09,val
1677902,22.033368,22.476992,21.982643,21.394213,21.701439,21.710030,21.126019,2.800000e+09,val
1677903,23.597499,23.518906,23.423641,24.135427,23.860683,23.430143,25.168051,3.700000e+10,val
1677904,21.882898,21.970409,21.842789,21.911989,21.924883,21.898005,22.174264,3.100000e+09,val


## Blends

| Blend | Models |
|---|---|
| structured | LightGBM, CatBoost, XGBoost, MLP |
| structured + TF-IDF | + the two models of notebook 06 |
| structured + BamiBERT | + BamiBERT |
| all | every model |

The weights come from a linear regression of the true log price on the models' predictions (validation set).
`positive=True` keeps every weight at zero or above.

In [3]:
STRUCTURED = ["LightGBM", "CatBoost", "XGBoost", "MLP"]
TFIDF = ["Ridge text + structured", "LightGBM + text"]
BLENDS = {
    "Blend: structured": STRUCTURED,
    "Blend: structured + TF-IDF": STRUCTURED + TFIDF,
    "Blend: structured + BamiBERT": STRUCTURED + ["BamiBERT"],
    "Blend: all": STRUCTURED + TFIDF + ["BamiBERT"],
}


def evaluate(y_true, y_pred):
    return {
        "MAE (bn VND)": mean_absolute_error(y_true, y_pred) / 1e9,
        "RMSE (bn VND)": root_mean_squared_error(y_true, y_pred) / 1e9,
        "R2": r2_score(y_true, y_pred),
        "MAPE (%)": 100 * mean_absolute_percentage_error(y_true, y_pred),
    }


results, weights = {}, {}

# each model on its own
for name in STRUCTURED + TFIDF + ["BamiBERT"]:
    results[name] = evaluate(test["price"], np.expm1(test[name]))

# weighted averages
for name, columns in BLENDS.items():
    blend = LinearRegression(positive=True).fit(val[columns], np.log1p(val["price"]))
    results[name] = evaluate(test["price"], np.expm1(blend.predict(test[columns])))
    weights[name] = pd.Series(blend.coef_, index=columns)

## Results

In [4]:
table = pd.DataFrame(results).T

os.makedirs("../reports/metrics", exist_ok=True)
table.to_csv("../reports/metrics/09_ensemble.csv")

table.style.format("{:,.3f}")

,MAE (bn VND),RMSE (bn VND),R2,MAPE (%)
LightGBM,4.464,18.189,0.739,26.240
CatBoost,5.242,21.065,0.650,31.664
XGBoost,4.485,18.401,0.733,26.418
MLP,4.620,19.264,0.707,26.832
Ridge text + structured,4.935,21.606,0.632,28.090
LightGBM + text,4.223,17.317,0.764,25.050
BamiBERT,4.355,17.353,0.763,26.856
Blend: structured,4.241,17.357,0.762,25.898
Blend: structured + TF-IDF,4.025,17.020,0.772,24.309
Blend: structured + BamiBERT,3.699,16.283,0.791,21.936


### Weights

A weight near 0 means the model adds nothing the other models in the blend do not already know.

In [5]:
pd.DataFrame(weights).T.style.format("{:.2f}", na_rep="")

,BamiBERT,CatBoost,LightGBM,LightGBM + text,MLP,Ridge text + structured,XGBoost
Blend: structured,,0.00,0.35,,0.31,,0.35
Blend: structured + TF-IDF,,0.00,0.00,0.49,0.10,0.21,0.21
Blend: structured + BamiBERT,0.46,0.00,0.17,,0.15,,0.21
Blend: all,0.41,0.00,0.01,0.23,0.10,0.06,0.19


### How to read this

- **structured** vs **structured + TF-IDF / + BamiBERT**: the difference is what the text adds.
- **+ TF-IDF** vs **+ BamiBERT**: does the language model get more out of the text than word counts?